In [2]:
from __future__ import annotations

from abc import ABC, abstractmethod
from datetime import date
from enum import Enum


class StatusContrato(Enum):
    ATIVO = "ativo"
    FINALIZADO = "finalizado"
    CANCELADO = "cancelado"


class Veiculo(ABC):
    def __init__(self, placa: str, modelo: str, ano: int, valor_diaria: float):
        self.placa = placa
        self.modelo = modelo
        self.ano = ano
        self.valor_diaria = valor_diaria
        self.contrato_ativo: Contrato | None = None

    def esta_disponivel(self) -> bool:
        return self.contrato_ativo is None

    @abstractmethod
    def descricao(self) -> str: ...

    def __str__(self) -> str:
        return self.descricao()


class Carro(Veiculo):
    def __init__(self, placa: str, modelo: str, ano: int, valor_diaria: float, numero_portas: int):
        super().__init__(placa, modelo, ano, valor_diaria)
        self.numero_portas = numero_portas

    def descricao(self) -> str:
        return f"Carro {self.modelo} ({self.placa}) - {self.numero_portas} portas"


class Moto(Veiculo):
    def __init__(self, placa: str, modelo: str, ano: int, valor_diaria: float, cilindradas: int):
        super().__init__(placa, modelo, ano, valor_diaria)
        self.cilindradas = cilindradas

    def descricao(self) -> str:
        return f"Moto {self.modelo} ({self.placa}) - {self.cilindradas}cc"


class Caminhao(Veiculo):
    def __init__(self, placa: str, modelo: str, ano: int, valor_diaria: float, capacidade_carga: float):
        super().__init__(placa, modelo, ano, valor_diaria)
        self.capacidade_carga = capacidade_carga

    def descricao(self) -> str:
        return f"Caminhão {self.modelo} ({self.placa}) - {self.capacidade_carga}t"


class Cliente(ABC):
    def __init__(self, nome: str, documento: str, telefone: str):
        self.nome = nome
        self.documento = documento
        self.telefone = telefone

    @abstractmethod
    def tipo_documento(self) -> str: ...

    def __str__(self) -> str:
        return f"{self.nome} ({self.tipo_documento()}: {self.documento})"


class PessoaFisica(Cliente):
    def tipo_documento(self) -> str:
        return "CPF"


class PessoaJuridica(Cliente):
    def tipo_documento(self) -> str:
        return "CNPJ"


class Condutor:
    def __init__(self, nome: str, cnh: str):
        self.nome = nome
        self.cnh = cnh

    def cnh_valida(self) -> bool:
        return self.cnh.isdigit() and len(self.cnh) == 11

    def __str__(self) -> str:
        return f"{self.nome} (CNH {self.cnh})"


class Contrato:
    """Composição com Condutor: o contrato instancia o condutor e o destrói ao ser excluído."""

    def __init__(
        self,
        cliente: Cliente,
        veiculo: Veiculo,
        data_inicio: date,
        data_termino_prevista: date,
        nome_condutor: str,
        cnh_condutor: str,
    ):
        if not veiculo.esta_disponivel():
            raise ValueError(f"O veículo {veiculo.placa} já possui contrato ativo.")
        if data_termino_prevista < data_inicio:
            raise ValueError("A data de término não pode ser anterior à data de início.")

        self.cliente = cliente
        self.veiculo = veiculo
        self.data_inicio = data_inicio
        self.data_termino_prevista = data_termino_prevista
        self.status = StatusContrato.ATIVO
        self.condutor: Condutor | None = Condutor(nome_condutor, cnh_condutor)
        self.valor_total = self.calcular_valor_total()
        veiculo.contrato_ativo = self

    def calcular_valor_total(self) -> float:
        dias = max((self.data_termino_prevista - self.data_inicio).days, 1)
        return dias * self.veiculo.valor_diaria

    def finalizar(self) -> None:
        self._encerrar(StatusContrato.FINALIZADO)

    def cancelar(self) -> None:
        self._encerrar(StatusContrato.CANCELADO)

    def excluir(self) -> None:
        if self.veiculo.contrato_ativo is self:
            self.veiculo.contrato_ativo = None
        self.condutor = None

    def _encerrar(self, novo_status: StatusContrato) -> None:
        self.status = novo_status
        if self.veiculo.contrato_ativo is self:
            self.veiculo.contrato_ativo = None


def main() -> None:
    cliente = PessoaFisica("Ana Costa", "123.456.789-00", "(86) 99999-0000")
    carro = Carro("ABC1D23", "Onix", 2024, 150.0, 4)
    moto = Moto("XYZ9K88", "CG 160", 2023, 80.0, 160)

    contrato = Contrato(
        cliente, carro, date(2026, 10, 5), date(2026, 10, 10),
        nome_condutor="Carlos Costa", cnh_condutor="12345678901",
    )

    print(f"Cliente: {contrato.cliente}")
    print(f"Veículo: {contrato.veiculo}")
    print(f"Condutor: {contrato.condutor} | CNH válida: {contrato.condutor.cnh_valida()}")
    print(f"Valor total: R$ {contrato.valor_total:.2f} | Status: {contrato.status.value}")

    print("\n--- Regra: veículo não pode ter dois contratos ativos ---")
    try:
        Contrato(cliente, carro, date(2026, 10, 6), date(2026, 10, 8), "Outro", "10987654321")
    except ValueError as erro:
        print(f"Bloqueado: {erro}")

    print("\n--- Excluindo o contrato (composição) ---")
    condutor_referencia = contrato.condutor
    contrato.excluir()
    print(f"Condutor no contrato: {contrato.condutor}")
    print(f"Veículo disponível novamente: {carro.esta_disponivel()}")
    print(f"Cliente continua existindo: {cliente}")
    print(f"Outro veículo intocado: {moto}")
    del condutor_referencia


if __name__ == "__main__":
    main()

Cliente: Ana Costa (CPF: 123.456.789-00)
Veículo: Carro Onix (ABC1D23) - 4 portas
Condutor: Carlos Costa (CNH 12345678901) | CNH válida: True
Valor total: R$ 750.00 | Status: ativo

--- Regra: veículo não pode ter dois contratos ativos ---
Bloqueado: O veículo ABC1D23 já possui contrato ativo.

--- Excluindo o contrato (composição) ---
Condutor no contrato: None
Veículo disponível novamente: True
Cliente continua existindo: Ana Costa (CPF: 123.456.789-00)
Outro veículo intocado: Moto CG 160 (XYZ9K88) - 160cc
